In [1]:
import pandas as pd

from evaluation.config import PROJECT_DIR
from granuscore.loader import JSONLineReader

reader = JSONLineReader()

In [2]:
def abbreviate(name):
    repl = {
        "scope_": "",
        "document": "doc",
        "sentence": "sent",
        "spooling_": "",
        "pooling": "pool",
        "doc_mean_": "doc_",
        "lower_quantile_mean": "lqm",
        "stailq": "stq",
        "tailq_": "",
    }

    for k, v in repl.items():
        name = name.replace(k, v)

    name = name.replace("_", "-")
    if "stq-" in name:
        stq = name.split("stq-")[1].split("-")[0]
        name = name.replace("sent-lqm-", f"sent-lqm-{stq}-")
        name = name.replace(f"-stq-{stq}", "")
    return name

stats = []

evaluations = {
    'news': reader.read(f"{PROJECT_DIR}/data/news_articles/evaluation/stats-news-articles.jsonl"),
    'movie': reader.read(f"{PROJECT_DIR}/data/domain-agnostic/evaluation/movie-stats-domain-agnostic.jsonl"),
    'twitter': reader.read(f"{PROJECT_DIR}/data/domain-agnostic/evaluation/twitter-stats-domain-agnostic.jsonl"),
    'yelp': reader.read(f"{PROJECT_DIR}/data/domain-agnostic/evaluation/yelp-stats-domain-agnostic.jsonl")
}

sizes = {
    'movie': 920,
    'twitter': 984,
    'yelp': 845,
    'news': 573
}
total = sum(sizes.values())


for i in range(len(evaluations['news'])):
    n = evaluations['news'][i]
    t = evaluations['twitter'][i]
    y = evaluations['yelp'][i]
    m = evaluations['movie'][i]
    assert n['model'] == t['model'] == y['model'] == m['model']

    explained_deviance = 0
    for domain, size in sizes.items():
        if domain == 'movie':
            value = m['explained_deviance']['length_plus_granularity'] - m['explained_deviance']['length_only']
        elif domain == 'twitter':
            value = t['explained_deviance']['length_plus_granularity'] - t['explained_deviance']['length_only']
        elif domain == 'yelp':
            value = y['explained_deviance']['length_plus_granularity'] - y['explained_deviance']['length_only']
        elif domain == 'news':
            value = n['explained_deviance']['length_plus_granularity'] - n['explained_deviance']['length_only']
        else:
            raise ValueError
        explained_deviance += value * size
    explained_deviance = explained_deviance / total

    name = n['model']
    stats.append({
        'name': abbreviate(name),
        'explained_deviance': explained_deviance,
    })
stats = pd.DataFrame(stats)

In [3]:
stats

,name,explained_deviance
0,sent-sum-pool-sum,0.036724
1,sent-sum-pool-mean,0.067523
2,sent-sum-pool-lqm-0.1,0.066448
3,sent-sum-pool-lqm-0.3,0.072708
4,sent-sum-pool-lqm-0.5,0.071195
...,...,...
58,sent-weighted-mean-pool-lqm-0.1,0.091379
59,sent-weighted-mean-pool-lqm-0.3,0.094074
60,sent-weighted-mean-pool-lqm-0.5,0.095412
61,sent-weighted-mean-pool-min,0.086851


In [4]:
main_stats = stats.copy().rename(columns={
    "name": "Method",
    "explained_deviance": "$\Delta$ Expl. Deviance",
})

num_cols = main_stats.columns.drop("Method")
main_stats[num_cols] = (main_stats[num_cols] * 100).round(2)

# copy for formatting
main_stats_fmt = main_stats.copy()

for col in num_cols:
    # get indices of largest and second largest
    order = main_stats[col].sort_values(ascending=False).index
    best = order[0]
    second = order[1]

    for i in stats.index:
        val = main_stats.loc[i, col]

        if i == best:
            main_stats_fmt.loc[i, col] = f"\\textbf{{{val:.2f}}}"
        elif i == second:
            main_stats_fmt.loc[i, col] = f"\\textit{{{val:.2f}}}"
        else:
            main_stats_fmt.loc[i, col] = f"{val:.2f}"

# export latex
latex = main_stats_fmt.to_latex(index=False, escape=False)

print(latex)

\begin{tabular}{ll}
\toprule
Method & $\Delta$ Expl. Deviance \\
\midrule
sent-sum-pool-sum & 3.67 \\
sent-sum-pool-mean & 6.75 \\
sent-sum-pool-lqm-0.1 & 6.64 \\
sent-sum-pool-lqm-0.3 & 7.27 \\
sent-sum-pool-lqm-0.5 & 7.12 \\
sent-sum-pool-min & 6.10 \\
sent-sum-pool-max & 1.51 \\
sent-mean-pool-sum & 2.55 \\
sent-mean-pool-mean & 8.34 \\
sent-mean-pool-lqm-0.1 & 9.19 \\
sent-mean-pool-lqm-0.3 & 9.20 \\
sent-mean-pool-lqm-0.5 & 9.20 \\
sent-mean-pool-min & 8.76 \\
sent-mean-pool-max & 1.87 \\
sent-lqm-0.9-pool-sum & 2.55 \\
sent-lqm-0.8-pool-sum & 2.56 \\
sent-lqm-0.7-pool-sum & 2.50 \\
sent-lqm-0.9-pool-mean & 8.34 \\
sent-lqm-0.8-pool-mean & 8.33 \\
sent-lqm-0.7-pool-mean & 8.33 \\
sent-lqm-0.9-pool-lqm-0.1 & 9.19 \\
sent-lqm-0.9-pool-lqm-0.3 & 9.20 \\
sent-lqm-0.9-pool-lqm-0.5 & 9.20 \\
sent-lqm-0.8-pool-lqm-0.1 & 9.20 \\
sent-lqm-0.8-pool-lqm-0.3 & 9.24 \\
sent-lqm-0.8-pool-lqm-0.5 & 9.24 \\
sent-lqm-0.7-pool-lqm-0.1 & 9.23 \\
sent-lqm-0.7-pool-lqm-0.3 & 9.35 \\
sent-lqm-0.7-pool-

<>:3: SyntaxWarning: invalid escape sequence '\D'
<>:3: SyntaxWarning: invalid escape sequence '\D'
/var/folders/pf/gbn3rrqn3sn7212qgqt0n3rw0000gn/T/ipykernel_99420/1092152105.py:3: SyntaxWarning: invalid escape sequence '\D'
  "explained_deviance": "$\Delta$ Expl. Deviance",
/var/folders/pf/gbn3rrqn3sn7212qgqt0n3rw0000gn/T/ipykernel_99420/1092152105.py:26: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '3.67' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  main_stats_fmt.loc[i, col] = f"{val:.2f}"


In [5]:
import numpy as np
import pandas as pd

df = main_stats_fmt.copy()
df = df[["Method", "$\Delta$ Expl. Deviance"]].reset_index(drop=True)

n_blocks = 3  # number of method/score pairs per row

blocks = np.array_split(df, n_blocks)

# reset index in each block so concat aligns row-wise
blocks = [b.reset_index(drop=True) for b in blocks]

# rename columns so headers repeat nicely
blocks = [
    b.rename(columns={"Method": "Method", "Expl. Deviance": "$\Delta$ Expl. Dev."})
    for b in blocks
]

table_df = pd.concat(blocks, axis=1)

latex = table_df.to_latex(index=False, escape=False)
print(latex)

\begin{tabular}{llllll}
\toprule
Method & $\Delta$ Expl. Deviance & Method & $\Delta$ Expl. Deviance & Method & $\Delta$ Expl. Deviance \\
\midrule
sent-sum-pool-sum & 3.67 & sent-lqm-0.9-pool-lqm-0.3 & 9.20 & sent-max-pool-sum & 2.74 \\
sent-sum-pool-mean & 6.75 & sent-lqm-0.9-pool-lqm-0.5 & 9.20 & sent-max-pool-mean & 7.09 \\
sent-sum-pool-lqm-0.1 & 6.64 & sent-lqm-0.8-pool-lqm-0.1 & 9.20 & sent-max-pool-lqm-0.1 & 7.81 \\
sent-sum-pool-lqm-0.3 & 7.27 & sent-lqm-0.8-pool-lqm-0.3 & 9.24 & sent-max-pool-lqm-0.3 & 7.68 \\
sent-sum-pool-lqm-0.5 & 7.12 & sent-lqm-0.8-pool-lqm-0.5 & 9.24 & sent-max-pool-lqm-0.5 & 7.71 \\
sent-sum-pool-min & 6.10 & sent-lqm-0.7-pool-lqm-0.1 & 9.23 & sent-max-pool-min & 7.28 \\
sent-sum-pool-max & 1.51 & sent-lqm-0.7-pool-lqm-0.3 & 9.35 & sent-max-pool-max & 2.00 \\
sent-mean-pool-sum & 2.55 & sent-lqm-0.7-pool-lqm-0.5 & 9.29 & doc-pool-sum & 3.49 \\
sent-mean-pool-mean & 8.34 & sent-lqm-0.9-pool-min & 8.76 & doc-pool-mean & 8.56 \\
sent-mean-pool-lqm-0.1 & 9

<>:5: SyntaxWarning: invalid escape sequence '\D'
<>:16: SyntaxWarning: invalid escape sequence '\D'
<>:5: SyntaxWarning: invalid escape sequence '\D'
<>:16: SyntaxWarning: invalid escape sequence '\D'
/var/folders/pf/gbn3rrqn3sn7212qgqt0n3rw0000gn/T/ipykernel_99420/1499405463.py:5: SyntaxWarning: invalid escape sequence '\D'
  df = df[["Method", "$\Delta$ Expl. Deviance"]].reset_index(drop=True)
/var/folders/pf/gbn3rrqn3sn7212qgqt0n3rw0000gn/T/ipykernel_99420/1499405463.py:16: SyntaxWarning: invalid escape sequence '\D'
  b.rename(columns={"Method": "Method", "Expl. Deviance": "$\Delta$ Expl. Dev."})
/Users/lukasellinger/PycharmProjects/granuscore/.venv/lib/python3.12/site-packages/numpy/_core/fromnumeric.py:57: FutureWarning: 'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.
  return bound(*args, **kwds)
